In [1]:
# @title Установка и настройка Ollama
#!/usr/bin/env python3
"""
Скрипт 1/3: Установка и запуск Ollama в Google Colab
Не настраивает ngrok — этим занимается run_server.py
"""

import os
import subprocess
import time
import sys

# Конфигурация
OLLAMA_MODELS_DIR = '/content/drive/MyDrive/mafia/ollama_models'
REQUIRED_MODELS = ["gemma3:12b"]
OLLAMA_PORT = 11434


def print_step(step_num, message):
    print(f"\n{'='*60}")
    print(f"ШАГ {step_num}: {message}")
    print(f"{'='*60}\n")


def run_command(cmd, check=True, shell=False):
    try:
        if isinstance(cmd, str) and not shell:
            cmd = cmd.split()
        result = subprocess.run(cmd, shell=shell, check=check, capture_output=True, text=True)
        return result
    except subprocess.CalledProcessError as e:
        print(f"❌ Ошибка выполнения команды: {e}")
        if e.stdout:
            print(f"STDOUT: {e.stdout}")
        if e.stderr:
            print(f"STDERR: {e.stderr}")
        raise


def check_colab():
    try:
        from google.colab import drive
        return True
    except ImportError:
        print("⚠️  Скрипт предназначен для Google Colab")
        return False


def mount_drive():
    print_step(1, "Монтирование Google Drive")
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print("✅ Google Drive успешно смонтирован")
        return True
    except Exception as e:
        print(f"❌ Ошибка монтирования Drive: {e}")
        return False


def setup_ollama_environment():
    print_step(2, "Настройка окружения Ollama")
    os.environ['OLLAMA_MODELS'] = OLLAMA_MODELS_DIR
    os.environ['OLLAMA_HOST'] = '0.0.0.0'
    os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
    print(f"✅ Директория для моделей: {OLLAMA_MODELS_DIR}")
    print(f"✅ OLLAMA_HOST=0.0.0.0 (разрешены внешние подключения)")
    return True


def install_ollama():
    print_step(3, "Установка Ollama")

    if os.path.exists('/usr/local/bin/ollama'):
        print("✅ Ollama уже установлена")
        return True

    print("📦 Установка зависимостей...")
    run_command("apt-get update -qq", shell=True)
    run_command("apt-get install -y -qq zstd curl", shell=True)

    print("📦 Установка Ollama...")
    run_command("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

    result = run_command(["ollama", "--version"], check=False)
    if result.returncode == 0:
        print(f"✅ Ollama установлена: {result.stdout.strip()}")
        return True
    else:
        print("❌ Ollama установлена, но не работает")
        return False


# Локальная директория для моделей — быстрый доступ без FUSE
OLLAMA_MODELS_LOCAL = '/content/ollama_models'


def copy_models_to_local():
    """Копирует модели с Drive в /content/ для быстрого доступа.
    Google Drive монтируется через FUSE — чтение весов идёт через сеть,
    первый inference занимает минуты. Локальная копия решает это."""
    import shutil

    if not os.path.exists(OLLAMA_MODELS_DIR):
        print("   Модели на Drive не найдены, пропускаю копирование")
        return False

    # Считаем размер на Drive
    total = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(OLLAMA_MODELS_DIR)
        for f in files
    )
    size_gb = total / (1024 ** 3)
    print(f"   Копирую модели с Drive ({size_gb:.1f} GB) в {OLLAMA_MODELS_LOCAL}...")

    if os.path.exists(OLLAMA_MODELS_LOCAL):
        shutil.rmtree(OLLAMA_MODELS_LOCAL)
    shutil.copytree(OLLAMA_MODELS_DIR, OLLAMA_MODELS_LOCAL)
    print(f"✅ Модели скопированы в {OLLAMA_MODELS_LOCAL}")
    return True


def start_ollama_service():
    print_step(4, "Запуск Ollama сервиса")

    result = run_command(["pgrep", "-f", "ollama serve"], check=False)
    if result.returncode == 0:
        print("🔄 Перезапуск Ollama с правильными настройками...")
        run_command(["pkill", "-f", "ollama serve"], check=False)
        time.sleep(2)

    print(f"🚀 Запуск Ollama (OLLAMA_HOST=0.0.0.0)...")
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0'
    env['OLLAMA_MODELS'] = OLLAMA_MODELS_DIR

    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    for i in range(15):
        time.sleep(1)
        result = run_command(
            ["curl", "-s", f"http://localhost:{OLLAMA_PORT}/api/tags"],
            check=False
        )
        if result.returncode == 0:
            print(f"✅ Ollama запущена на порту {OLLAMA_PORT}")
            return True
        print(f"   Ожидание... ({i+1}/15)")

    print("⚠️  Ollama запущена, но HTTP-проверка не прошла")
    return True


def check_model_installed(model_name):
    """Проверяет что модель установлена и имеет нормальный размер (не битая)"""
    try:
        result = run_command(["ollama", "list"], check=False)
        if result.returncode != 0 or model_name not in result.stdout:
            return False

        # Проверяем размер blobs на Drive — битая модель занимает мало места
        blobs_dir = os.path.join(OLLAMA_MODELS_DIR, "blobs")  # проверяем Drive — там эталон
        if not os.path.exists(blobs_dir):
            return False

        total_size = sum(
            os.path.getsize(os.path.join(blobs_dir, f))
            for f in os.listdir(blobs_dir)
            if os.path.isfile(os.path.join(blobs_dir, f))
        )
        size_gb = total_size / (1024 ** 3)

        # qwen2.5:14b весит ~9 GB — если меньше 5 GB считаем битой
        if size_gb < 5.0:
            print(f"   ⚠️  Модель битая: {size_gb:.1f} GB (ожидается >5 GB) — удаляю и скачиваю заново")
            run_command(["ollama", "rm", model_name], check=False)
            return False

        print(f"   Размер моделей на Drive: {size_gb:.1f} GB — OK")
        return True
    except Exception as e:
        print(f"   Ошибка проверки модели: {e}")
        return False


def download_models():
    print_step(5, "Загрузка моделей Ollama")

    for model in REQUIRED_MODELS:
        print(f"\n📥 Проверка модели: {model}")
        if check_model_installed(model):
            print(f"✅ Модель {model} уже установлена")
        else:
            print(f"📥 Загрузка {model}... (может занять несколько минут)")
            run_command(["ollama", "pull", model], check=True)
            print(f"✅ Модель {model} загружена")

    print("\n✅ Все модели готовы")
    return True


def warmup_model(model_name):
    """Прогревает модель — первый запрос загружает веса в VRAM.
    Без прогрева первый реальный inference занимает 60-70 секунд."""
    print(f"🔥 Прогрев модели {model_name} (загрузка весов в VRAM)...")
    result = run_command(
        ["curl", "-s", "-X", "POST", f"http://localhost:{OLLAMA_PORT}/api/chat",
         "-H", "Content-Type: application/json",
         "-d", f'{{"model":"{model_name}","messages":[{{"role":"user","content":"hi"}}],"stream":false}}'],
        check=False
    )
    if result.returncode == 0:
        print(f"✅ Модель прогрета — следующие запросы будут быстрыми")
    else:
        print(f"⚠️  Прогрев не удался: {result.stderr[:200]}")


def verify_ollama():
    print_step(6, "Проверка работы Ollama")
    result = run_command(["ollama", "list"], check=False)
    if result.returncode == 0:
        print("✅ Ollama работает")
        print("\n📋 Установленные модели:")
        print(result.stdout)
        for model in REQUIRED_MODELS:
            warmup_model(model)
        return True
    else:
        print("❌ Ollama не отвечает")
        return False


def main():
    print("\n" + "="*60)
    print("🚀 ШАГ 1/3: УСТАНОВКА OLLAMA")
    print("="*60 + "\n")

    is_colab = check_colab()

    steps = [
        ("Монтирование Drive", mount_drive if is_colab else lambda: True),
        ("Настройка окружения", setup_ollama_environment),
        ("Установка Ollama",   install_ollama),
        ("Запуск сервиса",     start_ollama_service),
        ("Загрузка моделей",   download_models),
        ("Проверка",           verify_ollama),
    ]

    for name, func in steps:
        try:
            if not func():
                print(f"⚠️  Шаг '{name}' завершился с предупреждением")
        except Exception as e:
            print(f"❌ Шаг '{name}' завершился с ошибкой: {e}")
            print("   Продолжаю выполнение...")

    print("\n" + "="*60)
    print("✅ OLLAMA ГОТОВА")
    print("="*60)
    print(f"\n   Ollama слушает на: http://localhost:{OLLAMA_PORT}")
    print(f"   Теперь запустите: setup_whisper_colab.py")
    print(f"   Затем запустите:  run_server.py\n")


if __name__ == "__main__":
    main()


🚀 ШАГ 1/3: УСТАНОВКА OLLAMA


ШАГ 1: Монтирование Google Drive

Mounted at /content/drive
✅ Google Drive успешно смонтирован

ШАГ 2: Настройка окружения Ollama

✅ Директория для моделей: /content/drive/MyDrive/mafia/ollama_models
✅ OLLAMA_HOST=0.0.0.0 (разрешены внешние подключения)

ШАГ 3: Установка Ollama

📦 Установка зависимостей...
📦 Установка Ollama...
✅ Ollama установлена: Warning: could not connect to a running Ollama instance

ШАГ 4: Запуск Ollama сервиса

🚀 Запуск Ollama (OLLAMA_HOST=0.0.0.0)...
✅ Ollama запущена на порту 11434

ШАГ 5: Загрузка моделей Ollama


📥 Проверка модели: gemma3:12b
   Размер моделей на Drive: 7.6 GB — OK
✅ Модель gemma3:12b уже установлена

✅ Все модели готовы

ШАГ 6: Проверка работы Ollama

✅ Ollama работает

📋 Установленные модели:
NAME          ID              SIZE      MODIFIED       
gemma3:12b    f4031aab637d    8.1 GB    46 minutes ago    

🔥 Прогрев модели gemma3:12b (загрузка весов в VRAM)...
✅ Модель прогрета — следующие запросы будут быстр

In [4]:
# @title Установка и настройка Whisper
#!/usr/bin/env python3
"""
Скрипт 2/3: Установка whisper.cpp и загрузка модели в Google Colab
Бинарник, все .so библиотеки и модель сохраняются на Google Drive.
"""

import os
import subprocess
import sys
import shutil

# Конфигурация
WHISPER_MODEL        = "ggml-medium.bin"
WHISPER_MODELS_DIR   = "/content/drive/MyDrive/mafia/whisper_models"
WHISPER_MODEL_PATH   = f"{WHISPER_MODELS_DIR}/{WHISPER_MODEL}"

# Файлы на Drive
WHISPER_BINARY_DRIVE = f"{WHISPER_MODELS_DIR}/whisper-cli"
WHISPER_LIBS_DRIVE   = f"{WHISPER_MODELS_DIR}/libs"  # папка со всеми .so

# Рабочие пути в сессии
WHISPER_BINARY_LOCAL = "/content/whisper.cpp/build/bin/whisper-cli"
WHISPER_LIBS_LOCAL   = "/content/whisper.cpp/build/libs"  # куда восстанавливаем .so

WHISPER_BINARY_CANDIDATES = [
    WHISPER_BINARY_LOCAL,
    "/content/whisper.cpp/build/bin/main",
    "/content/whisper.cpp/main",
]


def print_step(step_num, message):
    print(f"\n{'='*60}")
    print(f"ШАГ {step_num}: {message}")
    print(f"{'='*60}\n")


def run_command(cmd, check=True, shell=False):
    try:
        if isinstance(cmd, str) and not shell:
            cmd = cmd.split()
        result = subprocess.run(cmd, shell=shell, check=check, capture_output=True, text=True)
        return result
    except subprocess.CalledProcessError as e:
        print(f"❌ Ошибка: {e}")
        if e.stdout: print(f"STDOUT: {e.stdout}")
        if e.stderr: print(f"STDERR: {e.stderr}")
        raise


def find_all_libs():
    """Находит все .so файлы whisper после компиляции"""
    result = subprocess.run(
        ['find', '/content/whisper.cpp/build', '-name', '*.so*', '-type', 'f'],
        capture_output=True, text=True
    )
    return [p.strip() for p in result.stdout.splitlines() if p.strip()]


def register_libs_dir(libs_dir):
    """Регистрирует директорию со всеми .so в ldconfig"""
    try:
        with open('/etc/ld.so.conf.d/whisper.conf', 'w') as f:
            f.write(libs_dir + '\n')
        subprocess.run(['ldconfig'], check=False, capture_output=True)
        libs = os.listdir(libs_dir) if os.path.exists(libs_dir) else []
        print(f"✅ Зарегистрировано {len(libs)} библиотек из {libs_dir}: {libs}")
    except Exception as e:
        print(f"⚠️  ldconfig не удался: {e}")


def verify_binary(binary_path):
    """Проверяет что бинарник реально запускается"""
    result = subprocess.run([binary_path, '--help'], capture_output=True, text=True)
    ok = result.returncode == 0 or 'usage' in (result.stdout + result.stderr).lower()
    if ok:
        print(f"✅ Бинарник работает: {binary_path}")
    else:
        print(f"❌ Бинарник не запускается:\n{result.stderr[:300]}")
    return ok


def find_compiled_binary():
    for path in WHISPER_BINARY_CANDIDATES:
        if os.path.exists(path) and os.access(path, os.X_OK):
            return path
    return None


def mount_drive():
    print_step(1, "Монтирование Google Drive")
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print("✅ Google Drive смонтирован")
        return True
    except ImportError:
        print("⚠️  Не в Colab, пропускаю монтирование Drive")
        return True
    except Exception as e:
        print(f"❌ Ошибка монтирования: {e}")
        return False


def restore_binary_from_drive():
    """Восстанавливает бинарник + все .so с Drive"""
    if not os.path.exists(WHISPER_BINARY_DRIVE):
        return None

    size_mb = os.path.getsize(WHISPER_BINARY_DRIVE) / 1024 / 1024
    print(f"💾 Найден бинарник на Drive ({size_mb:.1f} MB) — восстанавливаю...")

    # Бинарник
    os.makedirs(os.path.dirname(WHISPER_BINARY_LOCAL), exist_ok=True)
    shutil.copy2(WHISPER_BINARY_DRIVE, WHISPER_BINARY_LOCAL)
    os.chmod(WHISPER_BINARY_LOCAL, 0o755)

    # Все .so библиотеки
    if os.path.exists(WHISPER_LIBS_DRIVE):
        os.makedirs(WHISPER_LIBS_LOCAL, exist_ok=True)
        for lib in os.listdir(WHISPER_LIBS_DRIVE):
            shutil.copy2(f"{WHISPER_LIBS_DRIVE}/{lib}", f"{WHISPER_LIBS_LOCAL}/{lib}")
        register_libs_dir(WHISPER_LIBS_LOCAL)
    else:
        print("⚠️  Папка libs не найдена на Drive")

    # Финальная проверка
    if not verify_binary(WHISPER_BINARY_LOCAL):
        print("❌ Бинарник с Drive не работает — нужна перекомпиляция")
        return None

    return WHISPER_BINARY_LOCAL


def save_binary_to_drive(binary_path):
    """Сохраняет бинарник + все .so на Drive"""
    os.makedirs(WHISPER_MODELS_DIR, exist_ok=True)

    shutil.copy2(binary_path, WHISPER_BINARY_DRIVE)
    size_mb = os.path.getsize(WHISPER_BINARY_DRIVE) / 1024 / 1024
    print(f"💾 Бинарник сохранён: {WHISPER_BINARY_DRIVE} ({size_mb:.1f} MB)")

    libs = find_all_libs()
    if libs:
        os.makedirs(WHISPER_LIBS_DRIVE, exist_ok=True)
        for lib in libs:
            shutil.copy2(lib, f"{WHISPER_LIBS_DRIVE}/{os.path.basename(lib)}")
        print(f"💾 Сохранено {len(libs)} библиотек в {WHISPER_LIBS_DRIVE}:")
        for lib in libs:
            print(f"   {os.path.basename(lib)}")
        register_libs_dir(WHISPER_LIBS_DRIVE)
    else:
        print("⚠️  .so файлы не найдены после компиляции")


def install_whisper():
    print_step(2, "Установка whisper.cpp")

    # 1. Уже есть в сессии
    binary = find_compiled_binary()
    if binary:
        libs = find_all_libs()
        if libs:
            # Копируем в единую папку и регистрируем
            os.makedirs(WHISPER_LIBS_LOCAL, exist_ok=True)
            for lib in libs:
                shutil.copy2(lib, f"{WHISPER_LIBS_LOCAL}/{os.path.basename(lib)}")
            register_libs_dir(WHISPER_LIBS_LOCAL)
        if verify_binary(binary):
            print(f"✅ Whisper готов: {binary}")
            return binary
        print("⚠️  Бинарник найден но не работает, пробую Drive...")

    # 2. Восстанавливаем с Drive
    binary = restore_binary_from_drive()
    if binary:
        return binary

    # 3. Компилируем с нуля
    print("📦 Компиляция (только один раз, потом берётся с Drive)")
    run_command("apt-get update -qq", shell=True, check=False)
    run_command("apt-get install -y -qq build-essential cmake git", shell=True, check=False)

    if not os.path.exists("/content/whisper.cpp"):
        result = run_command(
            "git clone https://github.com/ggerganov/whisper.cpp.git /content/whisper.cpp",
            shell=True, check=False
        )
        if result.returncode != 0:
            print(f"❌ Ошибка клонирования:\n{result.stderr}")
            return None
    else:
        # Директория есть но сломана — чистим и клонируем заново
        print("   Директория существует но сборка не удалась — переклонирую...")
        shutil.rmtree("/content/whisper.cpp")
        result = run_command(
            "git clone https://github.com/ggerganov/whisper.cpp.git /content/whisper.cpp",
            shell=True, check=False
        )
        if result.returncode != 0:
            print(f"❌ Ошибка клонирования:\n{result.stderr}")
            return None

    cuda_available = os.path.exists("/usr/local/cuda") or os.path.exists("/usr/bin/nvcc")
    if cuda_available:
        print("📦 Компиляция с CUDA GPU...")
        print("   ⏳ 15–25 минут — только сейчас, следующие сессии восстановят с Drive за секунды")
        cmake_flags = "-DGGML_CUDA=ON"
    else:
        print("📦 Компиляция CPU (~3–5 минут)...")
        cmake_flags = ""

    original_dir = os.getcwd()
    try:
        build_dir = "/content/whisper.cpp/build"
        os.makedirs(build_dir, exist_ok=True)
        os.chdir(build_dir)

        r1 = run_command(f"cmake .. {cmake_flags}", shell=True, check=False)
        if r1.returncode != 0:
            print(f"❌ CMake не прошёл:\n{r1.stderr[-500:]}")
            return None

        r2 = run_command("cmake --build . --config Release -j$(nproc)", shell=True, check=False)
        if r2.returncode != 0:
            print(f"❌ Ошибка сборки:\n{r2.stderr[-1000:]}")
            return None

        binary = find_compiled_binary()
        if not binary:
            print("❌ Бинарник не найден после компиляции")
            return None

        print(f"✅ Whisper скомпилирован: {binary}")
        save_binary_to_drive(binary)
        return binary

    except Exception as e:
        print(f"❌ Ошибка компиляции: {e}")
        return None
    finally:
        os.chdir(original_dir)


def download_model():
    print_step(3, "Загрузка модели whisper")
    os.makedirs(WHISPER_MODELS_DIR, exist_ok=True)

    if os.path.exists(WHISPER_MODEL_PATH):
        size_mb = os.path.getsize(WHISPER_MODEL_PATH) / 1024 / 1024
        print(f"✅ Модель уже на Drive: {WHISPER_MODEL_PATH} ({size_mb:.1f} MB)")
        return True

    print(f"📥 Загрузка {WHISPER_MODEL} с HuggingFace...")
    url = f"https://huggingface.co/ggerganov/whisper.cpp/resolve/main/{WHISPER_MODEL}"
    tmp_path = f"/tmp/{WHISPER_MODEL}"
    result = run_command(f"wget -q --show-progress -O {tmp_path} {url}", shell=True, check=False)

    if result.returncode != 0 or not os.path.exists(tmp_path):
        print(f"❌ Ошибка загрузки:\n{result.stderr[-300:]}")
        return False

    size_mb = os.path.getsize(tmp_path) / 1024 / 1024
    if size_mb < 10:
        print(f"❌ Файл слишком мал ({size_mb:.1f} MB)")
        return False

    shutil.copy2(tmp_path, WHISPER_MODEL_PATH)
    os.remove(tmp_path)
    print(f"✅ Модель сохранена: {WHISPER_MODEL_PATH} ({size_mb:.1f} MB)")
    return True


def save_binary_path(binary_path):
    config_path = "/content/whisper_config.txt"
    with open(config_path, 'w') as f:
        f.write(binary_path)
    print(f"✅ Путь сохранён: {config_path}")


def main():
    print("\n" + "="*60)
    print("🚀 ШАГ 2/3: УСТАНОВКА WHISPER")
    print("="*60 + "\n")

    mount_drive()

    binary = install_whisper()
    if not binary:
        print("❌ Не удалось установить whisper")
        sys.exit(1)

    if not download_model():
        print("❌ Не удалось загрузить модель")
        sys.exit(1)

    save_binary_path(binary)

    print("\n" + "="*60)
    print("✅ WHISPER ГОТОВ")
    print("="*60)
    print(f"\n   Бинарник:  {binary}")
    print(f"   На Drive:  {WHISPER_BINARY_DRIVE}")
    print(f"   Модель:    {WHISPER_MODEL_PATH}")
    print(f"\n   Теперь запустите: run_server.py\n")


if __name__ == "__main__":
    main()


🚀 ШАГ 2/3: УСТАНОВКА WHISPER


ШАГ 1: Монтирование Google Drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive смонтирован

ШАГ 2: Установка whisper.cpp

💾 Найден бинарник на Drive (0.9 MB) — восстанавливаю...
⚠️  Папка libs не найдена на Drive
❌ Бинарник не запускается:
/content/whisper.cpp/build/bin/whisper-cli: error while loading shared libraries: libwhisper.so.1: cannot open shared object file: No such file or directory

❌ Бинарник с Drive не работает — нужна перекомпиляция
📦 Компиляция (только один раз, потом берётся с Drive)
   Директория существует но сборка не удалась — переклонирую...
📦 Компиляция с CUDA GPU...
   ⏳ 15–25 минут — только сейчас, следующие сессии восстановят с Drive за секунды
✅ Whisper скомпилирован: /content/whisper.cpp/build/bin/whisper-cli
💾 Бинарник сохранён: /content/drive/MyDrive/mafia/whisper_models/whisper-cli (0.9 MB)
💾 Сохранено 5 библиотек в /content/d

In [ ]:
# @title Единый Flask-прокси + один ngrok туннель для Ollama и Whisper
#!/usr/bin/env python3
"""
Скрипт 3/3: Единый Flask-прокси + один ngrok туннель для Ollama и Whisper
Маршруты:
  /api/*          → Ollama (localhost:11434)
  /whisper/file   → whisper.cpp (прямой вызов бинарника)
  /health         → общий healthcheck
"""

import os
import subprocess
import time
import sys
import json
import tempfile
import urllib.request
from flask import Flask, request, Response, jsonify, stream_with_context

# ──────────────────────────────────────────────
# Конфигурация
# ──────────────────────────────────────────────
OLLAMA_PORT   = 11434
FLASK_PORT    = 5000
LANGUAGE      = "ru"
WHISPER_MODEL = "ggml-medium.bin"
WHISPER_MODELS_DIR  = "/content/drive/MyDrive/mafia/whisper_models"
WHISPER_MODEL_PATH  = f"{WHISPER_MODELS_DIR}/{WHISPER_MODEL}"
WHISPER_CONFIG_FILE = "/content/whisper_config.txt"   # записывается setup_whisper_colab.py
NGROK_AUTH_TOKEN    = '3AoDR90lG7GchjAwnPxAIdAOisx_5HFZ54HKyxfYTHcuNqMYa'

WHISPER_BINARY_CANDIDATES = [
    "/content/whisper.cpp/build/bin/whisper-cli",
    "/content/whisper.cpp/build/bin/main",
    "/content/whisper.cpp/main",
]

app = Flask(__name__)


# ──────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────

def run_command(cmd, check=True, shell=False):
    if isinstance(cmd, str) and not shell:
        cmd = cmd.split()
    return subprocess.run(cmd, shell=shell, check=check, capture_output=True, text=True)


def find_whisper_binary():
    """Ищет бинарник whisper: сначала из файла, потом по стандартным путям"""
    if os.path.exists(WHISPER_CONFIG_FILE):
        with open(WHISPER_CONFIG_FILE) as f:
            path = f.read().strip()
        if path and os.path.exists(path) and os.access(path, os.X_OK):
            return path

    for path in WHISPER_BINARY_CANDIDATES:
        if os.path.exists(path) and os.access(path, os.X_OK):
            return path
    return None


def get_model_path(model=None):
    model = model or WHISPER_MODEL

    # Клиент может прислать абсолютный путь своей машины (например, /home/roman/.../ggml-medium.bin).
    # На Colab такого пути нет — берём только имя файла и ищем его локально.
    if os.path.isabs(model):
        model = os.path.basename(model)

    gdrive_path = f"{WHISPER_MODELS_DIR}/{model}"
    if os.path.exists(gdrive_path):
        return gdrive_path
    local_path = f"/content/whisper.cpp/models/{model}"
    if os.path.exists(local_path):
        return local_path
    return gdrive_path  # вернём путь на Drive — ошибка будет понятной


def ollama_is_running():
    try:
        result = run_command(
            ["curl", "-s", "--max-time", "2", f"http://localhost:{OLLAMA_PORT}/api/tags"],
            check=False
        )
        return result.returncode == 0
    except:
        return False


# ──────────────────────────────────────────────
# Маршруты: Ollama-прокси
# ──────────────────────────────────────────────

@app.route('/api/<path:path>', methods=['GET', 'POST', 'PUT', 'DELETE'])
def ollama_proxy(path):
    """Проксирует все /api/* запросы к Ollama через requests (корректный проброс тела и заголовков)"""
    import requests as req_lib

    target_url = f"http://localhost:{OLLAMA_PORT}/api/{path}"
    if request.query_string:
        target_url += '?' + request.query_string.decode()

    # Пробрасываем заголовки — исключаем hop-by-hop которые ломают проксирование
    headers = {
        k: v for k, v in request.headers
        if k.lower() not in ('host', 'content-length', 'transfer-encoding', 'connection')
    }

    try:
        resp = req_lib.request(
            method=request.method,
            url=target_url,
            headers=headers,
            data=request.get_data(),
            stream=True,
            timeout=300,
        )

        def generate():
            for chunk in resp.iter_content(chunk_size=1):  # минимальный chunk — отдаём сразу
                if chunk:
                    yield chunk

        return Response(
            stream_with_context(generate()),
            status=resp.status_code,
            content_type=resp.headers.get('Content-Type', 'application/json'),
            headers={
                'X-Accel-Buffering': 'no',   # отключаем буферизацию nginx если есть
                'Cache-Control': 'no-cache',
            }
        )
    except Exception as e:
        return jsonify({"error": f"Ollama proxy error: {str(e)}"}), 502


# ──────────────────────────────────────────────
# Маршруты: Whisper
# ──────────────────────────────────────────────

@app.route('/whisper/file', methods=['POST'])
def whisper_file():
    """Принимает аудио файл, возвращает транскрипцию в NDJSON"""
    if 'audio' not in request.files:
        return jsonify({"error": "no audio file provided"}), 400

    audio_file = request.files['audio']
    language   = request.form.get('language', LANGUAGE)
    model      = request.form.get('model', WHISPER_MODEL)

    binary = find_whisper_binary()
    if not binary:
        return jsonify({"error": "whisper binary not found, run setup_whisper_colab.py first"}), 500

    model_path = get_model_path(model)
    if not os.path.exists(model_path):
        return jsonify({"error": f"model not found: {model_path}"}), 500

    with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as tmp:
        audio_file.save(tmp.name)
        tmp_path = tmp.name

    try:
        result = subprocess.run(
            [binary, "-m", model_path, "-f", tmp_path,
             "--language", language, "--no-prints",
             "--split-on-word", "--max-len", "100"],
            capture_output=True, text=True, timeout=300
        )

        if result.returncode != 0:
            # Включаем stderr в ответ для диагностики
            detail = (result.stderr or result.stdout or "").strip()[-500:]
            return jsonify({"error": f"whisper failed: {detail}"}), 500

        lines = []
        for line in result.stdout.split('\n'):
            line = line.strip()
            if not line:
                continue
            if line.startswith('['):
                idx = line.find(']')
                if idx != -1:
                    line = line[idx+1:].strip()
            if line:
                lines.append({"text": line})

        def generate():
            for item in lines:
                yield json.dumps(item, ensure_ascii=False) + "\n"

        return Response(generate(), mimetype='application/x-ndjson')

    except subprocess.TimeoutExpired:
        return jsonify({"error": "whisper timeout"}), 500
    except Exception as e:
        return jsonify({"error": str(e)}), 500
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)


# ──────────────────────────────────────────────
# Маршруты: Health
# ──────────────────────────────────────────────

@app.route('/health', methods=['GET'])
def health():
    binary = find_whisper_binary()
    return jsonify({
        "status": "ok",
        "ollama": {
            "running": ollama_is_running(),
            "url": f"http://localhost:{OLLAMA_PORT}",
        },
        "whisper": {
            "binary": binary or "not found",
            "model": WHISPER_MODEL,
            "model_path": WHISPER_MODEL_PATH,
            "model_exists": os.path.exists(WHISPER_MODEL_PATH),
        },
    })


# ──────────────────────────────────────────────
# Ngrok
# ──────────────────────────────────────────────

def install_ngrok():
    if os.path.exists('/usr/local/bin/ngrok'):
        print("✅ ngrok уже установлен")
        return True

    print("📦 Установка ngrok...")
    result = run_command(
        "curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null "
        "&& echo 'deb https://ngrok-agent.s3.amazonaws.com buster main' | tee /etc/apt/sources.list.d/ngrok.list "
        "&& apt update -qq && apt install ngrok -y -qq",
        shell=True, check=False
    )
    if os.path.exists('/usr/local/bin/ngrok'):
        print("✅ ngrok установлен")
        return True
    print("❌ Не удалось установить ngrok")
    return False


def setup_ngrok_tunnel():
    print("\n" + "="*60)
    print("Настройка ngrok туннеля")
    print("="*60)

    if not install_ngrok():
        return None

    # Останавливаем старый туннель если есть
    run_command(["pkill", "-f", "ngrok"], check=False)
    time.sleep(1)

    if NGROK_AUTH_TOKEN:
        print("🔐 Авторизация ngrok...")
        run_command(["ngrok", "config", "add-authtoken", NGROK_AUTH_TOKEN], check=False)
    else:
        print("⚠️  NGROK_AUTH_TOKEN не задан")

    print(f"🚇 Запуск туннеля на порт {FLASK_PORT}...")
    subprocess.Popen(
        ["ngrok", "http", str(FLASK_PORT), "--log=stdout"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(4)

    try:
        req = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=5)
        data = json.loads(req.read().decode())
        tunnels = data.get("tunnels", [])
        if tunnels:
            public_url = tunnels[0].get("public_url", "")
            if public_url:
                return public_url
    except Exception as e:
        print(f"⚠️  Не удалось получить URL туннеля: {e}")
        print("   Проверьте вручную: http://localhost:4040")

    return None


# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────

def main():
    print("\n" + "="*60)
    print("🚀 ШАГ 3/3: ЕДИНЫЙ СЕРВЕР (OLLAMA + WHISPER)")
    print("="*60 + "\n")

    # Проверка зависимостей — останавливаемся если что-то не готово
    errors = []

    if not ollama_is_running():
        errors.append("❌ Ollama не запущена — выполните setup_ollama_colab.py (ячейка 1)")
    else:
        print(f"✅ Ollama работает на порту {OLLAMA_PORT}")

    binary = find_whisper_binary()
    if not binary:
        errors.append("❌ Whisper бинарник не найден — выполните setup_whisper_colab.py (ячейка 2)")
    else:
        print(f"✅ Whisper бинарник: {binary}")

    if not os.path.exists(WHISPER_MODEL_PATH):
        errors.append(f"❌ Модель Whisper не найдена: {WHISPER_MODEL_PATH} — выполните setup_whisper_colab.py (ячейка 2)")
    else:
        size_mb = os.path.getsize(WHISPER_MODEL_PATH) / 1024 / 1024
        print(f"✅ Модель Whisper: {WHISPER_MODEL_PATH} ({size_mb:.1f} MB)")

    if errors:
        print("\n" + "="*60)
        print("🛑 ЗАПУСК ПРЕРВАН — не все компоненты готовы:")
        print("="*60)
        for e in errors:
            print(f"   {e}")
        print("\n💡 Порядок запуска:")
        print("   1. %run setup_ollama_colab.py")
        print("   2. %run setup_whisper_colab.py")
        print("   3. %run run_server.py")
        return

    # Запуск ngrok
    public_url = setup_ngrok_tunnel()
    base_url = public_url or f"http://localhost:{FLASK_PORT}"

    print("\n" + "="*60)
    print("✅ СЕРВЕР ГОТОВ")
    print("="*60)
    print(f"\n🌐 Локальный URL:  http://localhost:{FLASK_PORT}")
    if public_url:
        print(f"🌐 Публичный URL:  {public_url}")
    else:
        print(f"⚠️  ngrok туннель не создан")

    print(f"\n📋 Обновите config.json:")
    print(f'   "ollama_base_url": "{base_url}"')
    print(f'   "whisper_url":     "{base_url}/whisper/file"')

    print(f"\n📋 Endpoints:")
    print(f"   GET  {base_url}/health")
    print(f"   ANY  {base_url}/api/*          → Ollama")
    print(f"   POST {base_url}/whisper/file   → Whisper")

    print(f"\n⚠️  Для API-запросов через ngrok добавляйте заголовок:")
    print(f'   ngrok-skip-browser-warning: true')
    print(f"\n⚠️  Сессия Colab должна оставаться активной!\n")

    # Запуск Flask
    print("🚀 Запуск Flask сервера...")
    # use_reloader=False важен — иначе Popen для ngrok/ollama запускается дважды
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, threaded=True,
            use_reloader=False)


if __name__ == "__main__":
    main()


🚀 ШАГ 3/3: ЕДИНЫЙ СЕРВЕР (OLLAMA + WHISPER)

✅ Ollama работает на порту 11434
✅ Whisper бинарник: /content/whisper.cpp/build/bin/whisper-cli
✅ Модель Whisper: /content/drive/MyDrive/mafia/whisper_models/ggml-medium.bin (1462.7 MB)

Настройка ngrok туннеля
📦 Установка ngrok...
✅ ngrok установлен
🔐 Авторизация ngrok...
🚇 Запуск туннеля на порт 5000...

✅ СЕРВЕР ГОТОВ

🌐 Локальный URL:  http://localhost:5000
🌐 Публичный URL:  https://gregoria-unpacified-kendra.ngrok-free.dev

📋 Обновите config.json:
   "ollama_base_url": "https://gregoria-unpacified-kendra.ngrok-free.dev"
   "whisper_url":     "https://gregoria-unpacified-kendra.ngrok-free.dev/whisper/file"

📋 Endpoints:
   GET  https://gregoria-unpacified-kendra.ngrok-free.dev/health
   ANY  https://gregoria-unpacified-kendra.ngrok-free.dev/api/*          → Ollama
   POST https://gregoria-unpacified-kendra.ngrok-free.dev/whisper/file   → Whisper

⚠️  Для API-запросов через ngrok добавляйте заголовок:
   ngrok-skip-browser-warning: true


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:34:10] "GET /api/tags HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:35:57] "POST /whisper/file HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:37:58] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:38:04] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:38:11] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:38:19] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:38:33] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:38:58] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Mar/2026 15:39:32] "P